# Phase 10-A: MCP サーバー構造化ツール拡張実験

## 目的
- 構造化処理ツール付き拡張 MCP サーバー vs 既存シンプル MCP サーバーの性能比較
- パイプライン方式 vs LLM エージェント方式のオーケストレーション比較

## 4 システム比較（2×2 マトリクス）
|  | パイプライン方式 | LLM エージェント方式 |
|--|----------------|-------------------|
| **拡張 MCP** | A: Enhanced+Pipeline | C: Enhanced+Agent |
| **既存 MCP** | B: Simple+Pipeline | D: Simple+Agent |

## ベースライン
Phase 9-C C2 (Qwen3-32B + C1 prompt): composite 70.4pt

---
## 1. 環境セットアップ

In [ ]:
# 必要パッケージのインストール
!pip install -q 'mcp[cli]>=1.9.0' httpx nest_asyncio transformers accelerate bitsandbytes torch

In [ ]:
import os
import sys
import json
import time
import asyncio
import nest_asyncio
from datetime import datetime
from pathlib import Path

nest_asyncio.apply()

# ============================================================
# プロジェクトセットアップ
# ============================================================

# Google Drive マウント
from google.colab import drive
drive.mount('/content/drive')

# プロジェクトパス（Drive 上の配置場所に合わせてください）
PROJECT_DIR = '/content/drive/MyDrive/experiments-local-llm'

# src/ を Python パスに追加
sys.path.insert(0, os.path.join(PROJECT_DIR, 'src'))

# パス確認
print(f'Project dir: {PROJECT_DIR}')
print(f'src/ exists: {os.path.exists(os.path.join(PROJECT_DIR, "src"))}')
print(f'mcp_client.py exists: {os.path.exists(os.path.join(PROJECT_DIR, "src", "mcp_client.py"))}')
print(f'Python path: {sys.path[:3]}')

In [ ]:
# 結果保存ディレクトリ
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results', 'phase10a')
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Results dir: {RESULTS_DIR}')

---
## 2. MCP サーバー接続テスト

In [ ]:
# ============================================================
# ngrok トンネル URL を設定してください
# ============================================================
MCP_SERVER_URL = 'https://YOUR-NGROK-URL.ngrok-free.app/mcp'  # ← ここを変更
# ============================================================

print(f'MCP Server URL: {MCP_SERVER_URL}')

In [ ]:
from mcp_client import MCPClientWrapper

mcp_client = MCPClientWrapper(MCP_SERVER_URL)

# 接続テスト
ping_result = asyncio.get_event_loop().run_until_complete(mcp_client.ping())
print(json.dumps(ping_result, indent=2, ensure_ascii=False))

if ping_result['connected']:
    print(f"\n接続成功! {ping_result['tool_count']}個のツールが利用可能")
    geo_tools = [t for t in ping_result['tools'] if t.startswith('geo_')]
    mapfan_tools = [t for t in ping_result['tools'] if t.startswith('mapfan_')]
    print(f'  構造化ツール (geo_*): {len(geo_tools)}個 — {geo_tools}')
    print(f'  既存ツール (mapfan_*): {len(mapfan_tools)}個')
else:
    print(f"\n接続失敗: {ping_result.get('error', 'unknown')}")
    print('ngrok トンネルが起動していることを確認してください。')

In [ ]:
# 構造化ツール単体テスト
async def test_structured_tools():
    print('=== geo_analyze_question ===')
    result = await mcp_client.call_tool('geo_analyze_question', {
        'question': '渋谷駅から最も近いカフェはどこですか？'
    })
    print(result[:500])
    print()

    print('=== geo_nearest_pois ===')
    result = await mcp_client.call_tool('geo_nearest_pois', {
        'station_name': '渋谷駅',
        'category': 'カフェ',
        'radius': 500,
        'top_n': 3
    })
    print(result[:500])
    print()

    print('=== geo_count_by_category ===')
    result = await mcp_client.call_tool('geo_count_by_category', {
        'station_name': '新宿駅',
        'radius': 500
    })
    print(result[:500])

asyncio.get_event_loop().run_until_complete(test_structured_tools())

---
## 3. モデルロード

In [ ]:
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 既存モデルがあれば解放
if 'model' in dir() and model is not None:
    del model
if 'tokenizer' in dir() and tokenizer is not None:
    del tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# GPU 情報
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
gpu_free_gb = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9 if torch.cuda.is_available() else 0
print(f'GPU: {gpu_name} (Total: {gpu_mem_gb:.0f} GB, Free: {gpu_free_gb:.0f} GB)')

# モデル候補（大きい順に試行）
# NOTE: Qwen3-32B は 4bit でも ~20GB 必要だが A100 40GB では不安定。
#       安全のため A100 80GB 以上に制限。
MODEL_CANDIDATES = [
    ('Qwen/Qwen3-32B', 70),              # A100 80GB のみ
    ('Qwen/Qwen2.5-14B-Instruct', 20),   # A100 40GB, L4 24GB
    ('Qwen/Qwen2.5-7B-Instruct', 10),    # T4 16GB
]

# 空きメモリに基づいて選択（前回ロード失敗時の残留メモリ考慮）
MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'
for name, min_gb in MODEL_CANDIDATES:
    if gpu_free_gb >= min_gb:
        MODEL_NAME = name
        break

print(f'Selected model: {MODEL_NAME}')

# 4bit 量子化設定
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

def load_model(model_name):
    print(f'Loading {model_name} ...')
    tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    mdl = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map='auto',
        trust_remote_code=True,
    )
    return mdl, tok

try:
    model, tokenizer = load_model(MODEL_NAME)
except (ValueError, RuntimeError, torch.cuda.OutOfMemoryError) as e:
    print(f'Failed to load {MODEL_NAME}: {e}')
    gc.collect()
    torch.cuda.empty_cache()
    # フォールバック: 失敗モデル以降の小さいモデルを試行
    found_failed = False
    loaded = False
    for name, min_gb in MODEL_CANDIDATES:
        if name == MODEL_NAME:
            found_failed = True
            continue
        if not found_failed:
            continue
        try:
            print(f'Trying fallback: {name} ...')
            MODEL_NAME = name
            model, tokenizer = load_model(MODEL_NAME)
            loaded = True
            break
        except (ValueError, RuntimeError, torch.cuda.OutOfMemoryError) as e2:
            print(f'Failed: {e2}')
            gc.collect()
            torch.cuda.empty_cache()
            continue
    if not loaded:
        MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'
        model, tokenizer = load_model(MODEL_NAME)

print(f'Model loaded: {MODEL_NAME}')
print(f'GPU Memory used: {torch.cuda.memory_allocated()/1e9:.1f} GB / {gpu_mem_gb:.0f} GB')

---
## 4. System A: Enhanced+Pipeline（構築 & クイックテスト）

In [ ]:
from mcp_enhanced_pipeline import EnhancedMCPPipeline

system_a = EnhancedMCPPipeline(mcp_client, model=model, tokenizer=tokenizer, debug=True)

# クイックテスト
test_questions = [
    '渋谷駅から最も近いカフェはどこですか？',
    '新宿駅周辺のコンビニの数を教えてください',
    '池袋駅の東側と西側でラーメン店はどちらが多いですか？',
]

for q in test_questions:
    print(f'\n{"="*60}')
    print(f'Q: {q}')
    result = asyncio.get_event_loop().run_until_complete(system_a.query(q))
    print(f'A: {result["answer"][:300]}...')
    print(f'Tool calls: {[tc["tool"] for tc in result["tool_calls"]]}')
    print(f'Time: {result["time_sec"]}s')

---
## 5. System B: Simple+Pipeline（構築 & クイックテスト）

In [ ]:
from mcp_simple_pipeline import SimpleMCPPipeline

system_b = SimpleMCPPipeline(mcp_client, model=model, tokenizer=tokenizer, debug=True)

for q in test_questions:
    print(f'\n{"="*60}')
    print(f'Q: {q}')
    result = asyncio.get_event_loop().run_until_complete(system_b.query(q))
    print(f'A: {result["answer"][:300]}...')
    print(f'Tool calls: {[tc["tool"] for tc in result["tool_calls"]]}')
    print(f'Time: {result["time_sec"]}s')

---
## 6. System C: Enhanced+Agent（構築 & クイックテスト）

In [ ]:
from mcp_agent_system import MCPAgentSystem

# System C: 全ツール（enhanced + 基本）
system_c = MCPAgentSystem(mcp_client, model=model, tokenizer=tokenizer, tool_filter=None, debug=True)

for q in test_questions[:2]:  # Agent は時間がかかるため 2 問のみ
    print(f'\n{"="*60}')
    print(f'Q: {q}')
    result = asyncio.get_event_loop().run_until_complete(system_c.query(q))
    print(f'A: {result["answer"][:300]}...')
    print(f'Iterations: {result["iterations"]}, Tool calls: {len(result["tool_calls"])}')
    print(f'Time: {result["time_sec"]}s')

---
## 7. System D: Simple+Agent（構築 & クイックテスト）

In [ ]:
# System D: 基本ツールのみ (geo_* を除外)
system_d = MCPAgentSystem(mcp_client, model=model, tokenizer=tokenizer, tool_filter='simple', debug=True)

for q in test_questions[:2]:
    print(f'\n{"="*60}')
    print(f'Q: {q}')
    result = asyncio.get_event_loop().run_until_complete(system_d.query(q))
    print(f'A: {result["answer"][:300]}...')
    print(f'Iterations: {result["iterations"]}, Tool calls: {len(result["tool_calls"])}')
    print(f'Time: {result["time_sec"]}s')

---
## 8. 主評価: System A/B — Variant A（130 ケース × 2）

In [ ]:
from test_cases_multi_area import ALL_MULTI_AREA_TEST_CASES
from evaluators_multi_area import MultiAreaEvaluator

test_cases = ALL_MULTI_AREA_TEST_CASES
print(f'テストケース数: {len(test_cases)}')

evaluator = MultiAreaEvaluator()

In [ ]:
# System A 評価 (Enhanced+Pipeline, Variant A)
import gc

async def evaluate_system(system, system_name, test_cases, evaluator, results_dir, checkpoint_interval=10):
    """システム評価を実行（チェックポイント対応）"""
    results = []
    checkpoint_file = os.path.join(results_dir, f'{system_name}_checkpoint.json')
    
    # チェックポイントから再開
    start_idx = 0
    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, 'r') as f:
            checkpoint = json.load(f)
        results = checkpoint.get('results', [])
        start_idx = len(results)
        print(f'チェックポイントから再開: {start_idx}/{len(test_cases)}')
    
    for i, tc in enumerate(test_cases[start_idx:], start_idx):
        print(f'[{system_name}] {i+1}/{len(test_cases)} — {tc.id}: {tc.prompt[:40]}...')
        
        try:
            start = time.time()
            query_result = await system.query(tc.prompt)
            elapsed = time.time() - start
            
            answer = query_result.get('answer', '')
            
            # 評価
            eval_result = evaluator.evaluate_single(
                test_case=tc,
                answer=answer,
                system_name=system_name,
                time_sec=elapsed,
            )
            
            result_dict = {
                'test_id': tc.id,
                'system_name': system_name,
                'answer': answer[:1000],
                'time_sec': round(elapsed, 2),
                'tool_calls': query_result.get('tool_calls', []),
                'keyword_hit_rate': eval_result.keyword_hit_rate if hasattr(eval_result, 'keyword_hit_rate') else 0,
                'success': eval_result.success if hasattr(eval_result, 'success') else False,
                'composite_score': eval_result.composite_score if hasattr(eval_result, 'composite_score') else 0,
            }
            results.append(result_dict)
            
            print(f'  Score: {result_dict["composite_score"]:.1f}, '
                  f'KW hit: {result_dict["keyword_hit_rate"]:.2f}, '
                  f'Time: {elapsed:.1f}s')
            
        except Exception as e:
            print(f'  ERROR: {e}')
            results.append({
                'test_id': tc.id,
                'system_name': system_name,
                'answer': f'ERROR: {e}',
                'time_sec': 0,
                'tool_calls': [],
                'keyword_hit_rate': 0,
                'success': False,
                'composite_score': 0,
            })
        
        # チェックポイント保存
        if (i + 1) % checkpoint_interval == 0:
            with open(checkpoint_file, 'w') as f:
                json.dump({'results': results, 'timestamp': datetime.now().isoformat()}, f, ensure_ascii=False, indent=2)
            print(f'  Checkpoint saved: {i+1}/{len(test_cases)}')
        
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    # 最終結果保存
    final_file = os.path.join(results_dir, f'{system_name}_results.json')
    with open(final_file, 'w') as f:
        json.dump({
            'system_name': system_name,
            'total_cases': len(test_cases),
            'results': results,
            'timestamp': datetime.now().isoformat(),
        }, f, ensure_ascii=False, indent=2)
    print(f'\n結果保存: {final_file}')
    
    return results

In [ ]:
# System A 評価実行
results_a = asyncio.get_event_loop().run_until_complete(
    evaluate_system(system_a, 'SystemA_Enhanced_Pipeline_VA', test_cases, evaluator, RESULTS_DIR)
)

In [ ]:
# System B 評価実行
results_b = asyncio.get_event_loop().run_until_complete(
    evaluate_system(system_b, 'SystemB_Simple_Pipeline_VA', test_cases, evaluator, RESULTS_DIR)
)

---
## 9. 主評価: System A/B — Variant B（130 ケース × 2）

In [ ]:
from test_cases_multi_area_v2 import get_all_test_cases_v2, get_variant_b_stats

test_cases_vb = get_all_test_cases_v2()
stats = get_variant_b_stats()
print(f'Variant B テストケース数: {len(test_cases_vb)}')
print(f'キーワード変更されたケース: {stats["cases_with_keyword_changes"]}/{stats["total_cases"]}')

In [ ]:
# System A Variant B
results_a_vb = asyncio.get_event_loop().run_until_complete(
    evaluate_system(system_a, 'SystemA_Enhanced_Pipeline_VB', test_cases_vb, evaluator, RESULTS_DIR)
)

In [ ]:
# System B Variant B
results_b_vb = asyncio.get_event_loop().run_until_complete(
    evaluate_system(system_b, 'SystemB_Simple_Pipeline_VB', test_cases_vb, evaluator, RESULTS_DIR)
)

---
## 10. 補足評価: System C/D — Variant A（130 ケース × 2）

In [ ]:
# System C (Enhanced+Agent)
results_c = asyncio.get_event_loop().run_until_complete(
    evaluate_system(system_c, 'SystemC_Enhanced_Agent_VA', test_cases, evaluator, RESULTS_DIR)
)

In [ ]:
# System D (Simple+Agent)
results_d = asyncio.get_event_loop().run_until_complete(
    evaluate_system(system_d, 'SystemD_Simple_Agent_VA', test_cases, evaluator, RESULTS_DIR)
)

---
## 11. 4 システム比較分析

In [ ]:
def compute_summary(results, system_name):
    """評価結果のサマリーを計算"""
    if not results:
        return None
    
    scores = [r.get('composite_score', 0) for r in results]
    kw_rates = [r.get('keyword_hit_rate', 0) for r in results]
    successes = [r.get('success', False) for r in results]
    times = [r.get('time_sec', 0) for r in results]
    
    return {
        'system': system_name,
        'n_cases': len(results),
        'composite_mean': sum(scores) / len(scores) if scores else 0,
        'composite_median': sorted(scores)[len(scores)//2] if scores else 0,
        'kw_hit_rate_mean': sum(kw_rates) / len(kw_rates) if kw_rates else 0,
        'success_rate': sum(successes) / len(successes) if successes else 0,
        'avg_time_sec': sum(times) / len(times) if times else 0,
    }


# 全結果ファイルを読み込み
all_summaries = []
for fname in sorted(os.listdir(RESULTS_DIR)):
    if fname.endswith('_results.json'):
        with open(os.path.join(RESULTS_DIR, fname), 'r') as f:
            data = json.load(f)
        summary = compute_summary(data.get('results', []), data.get('system_name', fname))
        if summary:
            all_summaries.append(summary)

# ベースライン追加
all_summaries.append({
    'system': 'Baseline (Phase9C-C2)',
    'n_cases': 130,
    'composite_mean': 70.4,
    'composite_median': 0,
    'kw_hit_rate_mean': 0,
    'success_rate': 0,
    'avg_time_sec': 0,
})

# 比較表
print(f'{"System":<40} {"Composite":>10} {"KW Hit":>8} {"Success":>8} {"Time(s)":>8}')
print('-' * 80)
for s in sorted(all_summaries, key=lambda x: x['composite_mean'], reverse=True):
    print(f'{s["system"]:<40} {s["composite_mean"]:>10.1f} {s["kw_hit_rate_mean"]:>8.3f} '
          f'{s["success_rate"]:>8.1%} {s["avg_time_sec"]:>8.1f}')

In [ ]:
# 寄与分離分析
def find_summary(summaries, keyword):
    for s in summaries:
        if keyword in s['system']:
            return s
    return None

sa = find_summary(all_summaries, 'SystemA')
sb = find_summary(all_summaries, 'SystemB')
sc = find_summary(all_summaries, 'SystemC')
sd = find_summary(all_summaries, 'SystemD')

print('=== 寄与分離分析 ===')
print()
if sa and sb:
    diff_ab = sa['composite_mean'] - sb['composite_mean']
    print(f'A vs B (Pipeline): 構造化ツールの寄与 = {diff_ab:+.1f}pt')
    print(f'  A (Enhanced+Pipeline): {sa["composite_mean"]:.1f}pt')
    print(f'  B (Simple+Pipeline):   {sb["composite_mean"]:.1f}pt')
print()
if sc and sd:
    diff_cd = sc['composite_mean'] - sd['composite_mean']
    print(f'C vs D (Agent): 構造化ツールの寄与 = {diff_cd:+.1f}pt')
    print(f'  C (Enhanced+Agent): {sc["composite_mean"]:.1f}pt')
    print(f'  D (Simple+Agent):   {sd["composite_mean"]:.1f}pt')
print()
if sa and sc:
    diff_ac = sa['composite_mean'] - sc['composite_mean']
    print(f'A vs C: Pipeline vs Agent (Enhanced) = {diff_ac:+.1f}pt')
if sb and sd:
    diff_bd = sb['composite_mean'] - sd['composite_mean']
    print(f'B vs D: Pipeline vs Agent (Simple) = {diff_bd:+.1f}pt')

---
## 12. 結果保存

In [ ]:
# 全サマリーを保存
summary_file = os.path.join(RESULTS_DIR, 'phase10a_summary.json')
with open(summary_file, 'w') as f:
    json.dump({
        'summaries': all_summaries,
        'timestamp': datetime.now().isoformat(),
        'model': 'Qwen3-32B-4bit',
        'mcp_server_url': MCP_SERVER_URL,
    }, f, ensure_ascii=False, indent=2)

print(f'サマリー保存: {summary_file}')
print(f'結果ディレクトリ: {RESULTS_DIR}')
print(f'\n全ファイル:')
for f in sorted(os.listdir(RESULTS_DIR)):
    print(f'  {f}')